# EDA and preprocessing

Computes deterministic corpus summaries from the frozen 150/30 passage file and saves the important summary under reports/tables.

**Status:** provisional development evidence; zero locked test queries used.


In [1]:
from pathlib import Path
import csv, hashlib, json, random, statistics
from collections import Counter
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
print(f"Project: {ROOT.name} | fixed seed: {SEED}")


Project: RAABTA_PROJECT_PORTABLE | fixed seed: 20250816


In [2]:
rows = [json.loads(line) for line in (ROOT / "data/processed/passages_150_30.jsonl").open(encoding="utf-8")]
domains = {}
for row in rows:
    domains[row["domain"]] = domains.get(row["domain"], 0) + 1
summary = {"passages": len(rows), "mean_tokens": round(statistics.fmean(row["token_count"] for row in rows), 3), "domains": domains}
target = ROOT / "reports/tables/notebook_eda_summary.json"
target.write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "passages": 16352,
  "mean_tokens": 135.92,
  "domains": {
    "geography": 2943,
    "history": 5644,
    "general": 2265,
    "culture": 1921,
    "pakistan": 2500,
    "science": 1079
  }
}


## Article and passage distributions


In [3]:
articles = [json.loads(line) for line in (ROOT / 'data/raw/wikipedia.jsonl').open(encoding='utf-8') if line.strip()]
passages = [json.loads(line) for line in (ROOT / 'data/processed/passages_150_30.jsonl').open(encoding='utf-8') if line.strip()]
print('Article domains:', dict(sorted(Counter(row['domain'] for row in articles).items())))
print('Passage domains:', dict(sorted(Counter(row['domain'] for row in passages).items())))
tokens = [row['token_count'] for row in passages]
print({'min': min(tokens), 'median': statistics.median(tokens), 'mean': round(statistics.fmean(tokens), 3), 'max': max(tokens), 'unique_articles': len({row['article_id'] for row in passages})})


Article domains: {'culture': 460, 'general': 900, 'geography': 938, 'history': 757, 'pakistan': 748, 'science': 197}
Passage domains: {'culture': 1921, 'general': 2265, 'geography': 2943, 'history': 5644, 'pakistan': 2500, 'science': 1079}
{'min': 31, 'median': 150.0, 'mean': 135.92, 'max': 150, 'unique_articles': 4000}


## Chunking comparison


In [4]:
manifest = json.loads((ROOT / 'artifacts/metadata/phase1_manifest.json').read_text(encoding='utf-8'))
for item in manifest['passage_variants']:
    print(Path(item['path']).name, '| passages=', item['passages'], '| mean_tokens=', item['tokens']['mean'], '| represented_articles=', item['represented_articles'])


passages_120_24.jsonl | passages= 20106 | mean_tokens= 111.34 | represented_articles= 4000
passages_150_30.jsonl | passages= 16352 | mean_tokens= 135.92 | represented_articles= 4000
passages_180_36.jsonl | passages= 13908 | mean_tokens= 158.81 | represented_articles= 4000


## Interpretation

Outputs above are measurements from frozen local artifacts. Important limitations must remain attached when reused in the report or viva.
